# FINAL trial — public-1000 E21 parent-expanded, E19 max704

Runs only the E21 GPU1 winner on all 1,000 public questions. The saved public top-12 order and E19 metadata-trained rank-8 adapter are reused; exact E00 spans are compacted and expanded within the same bounded article. Two T4 GPUs split 500/500. Public answers are null and never read. E19 public METEOR 0.5309 remains the rollback.

In [ ]:
from pathlib import Path
import hashlib, json, shutil, subprocess, sys
config_hits = list(Path('/kaggle/input').rglob('final-public-e23-parent-expanded-max704-v1.json'))
assert len(config_hits) == 1, config_hits
CONFIG = config_hits[0]
PROJECT_ROOT = CONFIG.parents[1]
PUBLIC_PATH = PROJECT_ROOT / 'data/raw/public-official.json'
def sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''): h.update(block)
    return h.hexdigest()
retrieval_hits, e00_hits, e19_hits, selection_hits = [], [], [], []
for path in Path('/kaggle/input').rglob('report.json'):
    try: report = json.loads(path.read_text(encoding='utf-8'))
    except Exception: continue
    if report.get('experiment_id') == 'FINAL-public1000-aiteam-qwen35-lora-v1': retrieval_hits.append(path.parent)
    if report.get('experiment_id') == 'E21-parent-context-dev200-v1': selection_hits.append(path.parent)
for path in Path('/kaggle/input').rglob('manifest.json'):
    if sha256(path) == '04efd3905ad6d2758461587ca68d7d70fa2c568855b78f0c762c7a37ba547b2e': e00_hits.append(path.parent)
for path in Path('/kaggle/input').rglob('adapter-final/complete.json'):
    try: complete = json.loads(path.read_text(encoding='utf-8'))
    except Exception: continue
    if complete.get('experiment_id') == 'E19-metadata-aware-lora-train5636-eval200-v1': e19_hits.append(path.parents[2])
assert len(retrieval_hits) == len(e00_hits) == len(e19_hits) == len(selection_hits) == 1, {'retrieval': retrieval_hits, 'e00': e00_hits, 'e19': e19_hits, 'selection': selection_hits}
RETRIEVAL_ARTIFACT, E00_ARTIFACT = retrieval_hits[0], e00_hits[0]
E19_ARTIFACT, SELECTION_ARTIFACT = e19_hits[0], selection_hits[0]
OUTPUT = Path('/kaggle/working/final-public1000-e23-parent-expanded-max704-v1')
RESUME_FROM = None  # Only a complete saved E23 output; leave None for same-session reruns.
APPROVALS = ['inventory-reviewed', 'experiment-plan-reviewed', 'weights-download-authorized', 'operator-selection-authorized', 'public-inference-authorized', 'submission-creation-authorized', 'e23-public-trial-authorized']
print({'retrieval': str(RETRIEVAL_ARTIFACT), 'e00': str(E00_ARTIFACT), 'e19_adapter': str(E19_ARTIFACT), 'e21_selection': str(SELECTION_ARTIFACT), 'output': str(OUTPUT), 'policy': 'E21 GPU1 parent-expanded', 'max_new_tokens': 704, 'gpu_partitions': [500, 500], 'rollback_meteor': 0.5309})

In [ ]:
if RESUME_FROM is not None:
    RESUME_FROM = Path(RESUME_FROM)
    assert RESUME_FROM.is_dir() and (RESUME_FROM / 'preflight.json').is_file()
    assert not OUTPUT.exists(), 'Do not merge saved and current checkpoints.'
    shutil.copytree(RESUME_FROM, OUTPUT)
OUTPUT.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=5.1,<6', 'accelerate>=1.10,<2', 'peft>=0.18,<1', 'nltk==3.7'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
import torch
assert torch.cuda.device_count() == 2, 'Choose GPU T4 x2'
print([torch.cuda.get_device_name(i) for i in range(2)])

In [ ]:
RUNNER = PROJECT_ROOT / 'scripts/run_final_public_e23_kaggle.py'
def phase_command(phase, rank=None):
    paths = {'project-root': PROJECT_ROOT, 'retrieval-artifact': RETRIEVAL_ARTIFACT, 'e00-artifact': E00_ARTIFACT, 'e19-artifact': E19_ARTIFACT, 'selection-artifact': SELECTION_ARTIFACT, 'public': PUBLIC_PATH, 'output': OUTPUT, 'config': CONFIG}
    command = [sys.executable, str(RUNNER), phase]
    for key, path in paths.items(): command.extend(['--' + key, str(path)])
    command.extend('--' + key for key in APPROVALS)
    if rank is not None: command.extend(['--worker-rank', str(rank), '--device', f'cuda:{rank}'])
    return command
def run_phase(phase, **kwargs):
    print('Running:', phase)
    subprocess.run(phase_command(phase, **kwargs), check=True)
run_phase('preflight')
run_phase('prepare-contexts')
print((OUTPUT / 'prepared-summary.json').read_text(encoding='utf-8'))

In [ ]:
# Both GPUs run the same E21 GPU1 policy on disjoint IDs: 500 each.
import os, time
worker_env = os.environ.copy(); worker_env['OMP_NUM_THREADS'] = '2'
processes = {}
try:
    for rank in range(2):
        print('Starting E23 public worker:', rank, f'cuda:{rank}')
        processes[rank] = subprocess.Popen(phase_command('generate-worker', rank=rank), env=worker_env)
    while any(process.poll() is None for process in processes.values()):
        failures = {rank: process.returncode for rank, process in processes.items() if process.poll() not in (None, 0)}
        if failures:
            for process in processes.values():
                if process.poll() is None: process.terminate()
            break
        time.sleep(2)
    exit_codes = {rank: process.wait() for rank, process in processes.items()}
finally:
    for process in processes.values():
        if process.poll() is None: process.terminate()
    for process in processes.values():
        if process.poll() is None: process.wait()
failures = {rank: code for rank, code in exit_codes.items() if code != 0}
assert not failures, failures
print('Both E23 workers completed: 1,000/1,000 answers.')

In [ ]:
run_phase('finalize')
report = json.loads((OUTPUT / 'report.json').read_text(encoding='utf-8'))
print(json.dumps(report, ensure_ascii=False, indent=2))
FINAL_ZIP = Path('/kaggle/working/submission.zip')
shutil.copy2(OUTPUT / 'submission.zip', FINAL_ZIP)
import zipfile
with zipfile.ZipFile(FINAL_ZIP) as archive: assert archive.namelist() == ['submission.json'], archive.namelist()
print({'status': 'VALID — READY TO SUBMIT', 'submission_zip': str(FINAL_ZIP), 'questions': 1000, 'rollback_meteor': 0.5309})
print('Submit submission.zip. The public METEOR is only available from the leaderboard; Save Version with Always Save Output.')